<a href="https://colab.research.google.com/github/Heptazero/ml-course-labs/blob/main/deep-learning/exp2_rnn_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 实验二：循环神经网络实践（RNN 与 LSTM）

## 1. 环境部署与测试

In [ ]:
%matplotlib inline
import collections
import math
import os
import random
import re
import time
import urllib.request
import torch
from torch import nn
from torch.nn import functional as F
from matplotlib import pyplot as plt

def try_gpu(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

device = try_gpu()
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Time Machine 数据集加载与文本预处理（第 8.2、8.3 节）

In [ ]:
def download_time_machine():
    os.makedirs('../data', exist_ok=True)
    fname = '../data/timemachine.txt'
    if not os.path.exists(fname):
        url = 'http://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt'
        urllib.request.urlretrieve(url, fname)
    return fname

def read_time_machine():
    with open(download_time_machine(), 'r', encoding='utf-8') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

def tokenize(lines, token='word'):
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        raise ValueError('Unknown token type: ' + token)

def count_corpus(tokens):
    if len(tokens) == 0 or isinstance(tokens[0], list):
        tokens = [token for line in tokens for token in line]
    return collections.Counter(tokens)

class Vocab:
    def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
        if tokens is None:
            tokens = []
        if reserved_tokens is None:
            reserved_tokens = []
        counter = count_corpus(tokens)
        self._token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.idx_to_token = ['<unk>'] + reserved_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(self.idx_to_token)}
        for token, freq in self._token_freqs:
            if freq < min_freq:
                break
            if token not in self.token_to_idx:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1

    def __len__(self):
        return len(self.idx_to_token)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(tokens, self.unk)
        return [self.__getitem__(token) for token in tokens]

    def to_tokens(self, indices):
        if not isinstance(indices, (list, tuple)):
            return self.idx_to_token[indices]
        return [self.idx_to_token[index] for index in indices]

    @property
    def unk(self):
        return 0

    @property
    def token_freqs(self):
        return self._token_freqs

def load_corpus_time_machine(max_tokens=-1):
    lines = read_time_machine()
    tokens = tokenize(lines, 'char')
    vocab = Vocab(tokens)
    corpus = [vocab[token] for line in tokens for token in line]
    if max_tokens > 0:
        corpus = corpus[:max_tokens]
    return corpus, vocab

lines = read_time_machine()
full_corpus, vocab = load_corpus_time_machine()
print('总行数:', len(lines))
print('第 0 行:', lines[0])
print('第 10 行:', lines[10])
print('全语料字符数:', len(full_corpus), '词表大小:', len(vocab))
print('前 10 个高频字符:', vocab.token_freqs[:10])

In [ ]:
def seq_data_iter_random(corpus, batch_size, num_steps):
    corpus = corpus[random.randint(0, num_steps - 1):]
    num_subseqs = (len(corpus) - 1) // num_steps
    initial_indices = list(range(0, num_subseqs * num_steps, num_steps))
    random.shuffle(initial_indices)
    def data(pos):
        return corpus[pos: pos + num_steps]
    num_batches = num_subseqs // batch_size
    for i in range(0, batch_size * num_batches, batch_size):
        initial_indices_per_batch = initial_indices[i: i + batch_size]
        X = [data(j) for j in initial_indices_per_batch]
        Y = [data(j + 1) for j in initial_indices_per_batch]
        yield torch.tensor(X), torch.tensor(Y)

def seq_data_iter_sequential(corpus, batch_size, num_steps):
    offset = random.randint(0, num_steps)
    num_tokens = ((len(corpus) - offset - 1) // batch_size) * batch_size
    Xs = torch.tensor(corpus[offset: offset + num_tokens])
    Ys = torch.tensor(corpus[offset + 1: offset + 1 + num_tokens])
    Xs, Ys = Xs.reshape(batch_size, -1), Ys.reshape(batch_size, -1)
    num_batches = Xs.shape[1] // num_steps
    for i in range(0, num_steps * num_batches, num_steps):
        X = Xs[:, i: i + num_steps]
        Y = Ys[:, i: i + num_steps]
        yield X, Y

class SeqDataLoader:
    def __init__(self, corpus, vocab, batch_size, num_steps, use_random_iter):
        self.data_iter_fn = seq_data_iter_random if use_random_iter else seq_data_iter_sequential
        self.corpus, self.vocab = corpus, vocab
        self.batch_size, self.num_steps = batch_size, num_steps

    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.batch_size, self.num_steps)

def load_data_time_machine(batch_size, num_steps, use_random_iter=False, max_tokens=10000):
    corpus, vocab = load_corpus_time_machine(max_tokens)
    data_iter = SeqDataLoader(corpus, vocab, batch_size, num_steps, use_random_iter)
    return data_iter, vocab

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps, max_tokens=10000)
test_iter = SeqDataLoader(full_corpus[10000:15000], vocab, batch_size, num_steps, use_random_iter=False)
X_sample, Y_sample = next(iter(train_iter))
print('X shape:', X_sample.shape, 'Y shape:', Y_sample.shape)

## 3. 通用模型封装、预测与训练评估函数（第 8.5、8.6 节）

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        super(RNNModel, self).__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size
        if not self.rnn.bidirectional:
            self.num_directions = 1
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            self.linear = nn.Linear(self.num_hiddens * 2, self.vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(torch.float32)
        Y, state = self.rnn(X, state)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, device, batch_size=1):
        if not isinstance(self.rnn, nn.LSTM):
            return torch.zeros((self.num_directions * self.rnn.num_layers,
                                batch_size, self.num_hiddens), device=device)
        else:
            return (torch.zeros((self.num_directions * self.rnn.num_layers,
                                 batch_size, self.num_hiddens), device=device),
                    torch.zeros((self.num_directions * self.rnn.num_layers,
                                 batch_size, self.num_hiddens), device=device))

def predict_ch8(prefix, num_preds, net, vocab, device):
    net.eval()
    state = net.begin_state(batch_size=1, device=device)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    with torch.no_grad():
        for y in prefix[1:]:
            _, state = net(get_input(), state)
            outputs.append(vocab[y])
        for _ in range(num_preds):
            y, state = net(get_input(), state)
            outputs.append(int(y.argmax(dim=1).item()))
    return ''.join([vocab.idx_to_token[i] for i in outputs])

def grad_clipping(net, theta):
    params = [p for p in net.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

def evaluate_perplexity(net, data_iter, device):
    net.eval()
    loss = nn.CrossEntropyLoss()
    state = None
    loss_sum, token_count = 0.0, 0
    with torch.no_grad():
        for X, Y in data_iter:
            if state is None:
                state = net.begin_state(batch_size=X.shape[0], device=device)
            y = Y.T.reshape(-1)
            X, y = X.to(device), y.to(device)
            y_hat, state = net(X, state)
            l = loss(y_hat, y.long()).mean()
            loss_sum += float(l) * y.numel()
            token_count += y.numel()
    return math.exp(loss_sum / token_count)

def train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter):
    net.train()
    state = None
    t0 = time.time()
    loss_sum, token_count = 0.0, 0
    for X, Y in train_iter:
        if state is None or use_random_iter:
            state = net.begin_state(batch_size=X.shape[0], device=device)
        else:
            if not isinstance(state, tuple):
                state.detach_()
            else:
                for s in state:
                    s.detach_()
        y = Y.T.reshape(-1)
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state)
        l = loss(y_hat, y.long()).mean()
        updater.zero_grad()
        l.backward()
        grad_clipping(net, 1)
        updater.step()
        loss_sum += float(l) * y.numel()
        token_count += y.numel()
    return math.exp(loss_sum / token_count), token_count / (time.time() - t0)

def train_ch8(net, train_iter, test_iter, vocab, lr, num_epochs, device, use_random_iter=False):
    loss = nn.CrossEntropyLoss()
    updater = torch.optim.SGD(net.parameters(), lr)
    history = {'epochs': [], 'train_ppl': [], 'speed': 0.0}
    speeds = []
    for epoch in range(num_epochs):
        ppl, speed = train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter)
        speeds.append(speed)
        if (epoch + 1) % 10 == 0:
            history['epochs'].append(epoch + 1)
            history['train_ppl'].append(ppl)
        if (epoch + 1) % 100 == 0:
            print(f'epoch {epoch + 1:3d}/{num_epochs}: train ppl {ppl:.2f} | {predict_ch8("time traveller", 50, net, vocab, device)}')
    test_ppl = evaluate_perplexity(net, test_iter, device)
    avg_speed = sum(speeds) / len(speeds)
    history['final_train_ppl'] = ppl
    history['test_ppl'] = test_ppl
    history['speed'] = avg_speed
    print(f'train perplexity {ppl:.2f}, test perplexity {test_ppl:.2f}, {avg_speed:.1f} tokens/sec on {str(device)}')
    print(predict_ch8('time traveller', 50, net, vocab, device))
    print(predict_ch8('traveller', 50, net, vocab, device))

    plt.figure(figsize=(6, 4))
    plt.plot(history['epochs'], history['train_ppl'], '-', label='train perplexity')
    plt.xlabel('epoch')
    plt.ylabel('perplexity')
    plt.xlim(10, num_epochs)
    plt.legend()
    plt.grid()
    plt.show()
    return history

## 4. 循环神经网络 RNN（第 8.6 节）

### 4.1 模型配置与张量形状检查

In [ ]:
num_hiddens = 256
rnn_layer = nn.RNN(len(vocab), num_hiddens)
state = torch.zeros((1, batch_size, num_hiddens))
X_dummy = torch.rand(size=(num_steps, batch_size, len(vocab)))
Y_dummy, state_new = rnn_layer(X_dummy, state)
print('RNN state shape:', state.shape)
print('RNN output shape:', Y_dummy.shape, 'new state shape:', state_new.shape)

rnn_net = RNNModel(rnn_layer, vocab_size=len(vocab)).to(device)
rnn_params = sum(p.numel() for p in rnn_net.parameters())
print(f'RNN parameters: {rnn_params:,}')
print('Before training:', predict_ch8('time traveller', 10, rnn_net, vocab, device))

### 4.2 训练与测试 RNN

In [ ]:
num_epochs, lr = 500, 1
rnn_hist = train_ch8(rnn_net, train_iter, test_iter, vocab, lr, num_epochs, device)

## 5. 长短期记忆网络 LSTM（第 9.2 节）

### 5.1 模型配置与张量形状检查

In [ ]:
lstm_layer = nn.LSTM(len(vocab), num_hiddens)
h0 = torch.zeros((1, batch_size, num_hiddens))
c0 = torch.zeros((1, batch_size, num_hiddens))
Y_lstm, (hn, cn) = lstm_layer(X_dummy, (h0, c0))
print('LSTM output shape:', Y_lstm.shape, 'hn shape:', hn.shape, 'cn shape:', cn.shape)

lstm_net = RNNModel(lstm_layer, vocab_size=len(vocab)).to(device)
lstm_params = sum(p.numel() for p in lstm_net.parameters())
print(f'LSTM parameters: {lstm_params:,}')
print('Before training:', predict_ch8('time traveller', 10, lstm_net, vocab, device))

### 5.2 训练与测试 LSTM

In [ ]:
lstm_hist = train_ch8(lstm_net, train_iter, test_iter, vocab, lr, num_epochs, device)

## 6. 实验结果对比与文本生成测试

In [ ]:
print(f'{"Model":<10} {"Params":>10} {"Train PPL":>12} {"Test PPL":>12} {"Speed (tok/s)":>16}')
print('-' * 64)
print(f'{"RNN":<10} {rnn_params:>10,} {rnn_hist["final_train_ppl"]:>12.2f} '
      f'{rnn_hist["test_ppl"]:>12.2f} {rnn_hist["speed"]:>16.1f}')
print(f'{"LSTM":<10} {lstm_params:>10,} {lstm_hist["final_train_ppl"]:>12.2f} '
      f'{lstm_hist["test_ppl"]:>12.2f} {lstm_hist["speed"]:>16.1f}')

plt.figure(figsize=(7, 4))
plt.plot(rnn_hist['epochs'], rnn_hist['train_ppl'], '-', label='RNN')
plt.plot(lstm_hist['epochs'], lstm_hist['train_ppl'], '--', label='LSTM')
plt.xlabel('epoch')
plt.ylabel('perplexity')
plt.title('RNN vs LSTM Training Perplexity')
plt.xlim(10, num_epochs)
plt.legend()
plt.grid()
plt.show()

In [ ]:
prefixes = ['time traveller', 'traveller', 'the time machine', 'dimension']
for prefix in prefixes:
    print(f'Prefix: "{prefix}"')
    print('  RNN :', predict_ch8(prefix, 60, rnn_net, vocab, device))
    print('  LSTM:', predict_ch8(prefix, 60, lstm_net, vocab, device))